# Benchmark parser M2 trên Colab CPU

Chọn Runtime → Change runtime type → CPU. Notebook dùng cùng snapshot Drive 73 paper/119 version với bản nộp 06/11, không gọi arXiv hoặc Semantic Scholar.

Tải lên hai file đi kèm: `arxiv_parser_runtime.zip` và `arxiv_parser_input.zip`. Chạy hết notebook để tải kết quả đo thật về máy. Kết quả này chỉ thuộc snapshot được nêu; không thay cho xử lý toàn dải.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, stat, tempfile, zipfile, subprocess
from google.colab import files

if Path('/dev/nvidia0').exists() or os.environ.get('COLAB_TPU_ADDR'):
    raise RuntimeError('Chọn runtime CPU rồi chạy lại.')
SOURCE_SHA256 = '4b743d0b281ec2a6d4a6a8adf1bc0506b2fb692b4cb54b258afe271525fd8eab'
INPUT_SHA256 = '53c6518fbec9a913058a482b8eabc7f3584df6225940c3b5c008dd316ea3ebb9'
uploaded = files.upload()
expected = {'arxiv_parser_runtime.zip': SOURCE_SHA256, 'arxiv_parser_input.zip': INPUT_SHA256}
for name, digest in expected.items():
    if name not in uploaded or hashlib.sha256(uploaded[name]).hexdigest() != digest:
        raise RuntimeError('File thiếu hoặc không đúng hash: ' + name)
    Path('/content', name).write_bytes(uploaded[name])
print('Hai gói đã khớp SHA-256.')


In [ ]:
def allowed_path(name: str) -> bool:
    path = Path(name)
    return (path.as_posix() == name and not path.is_absolute() and ".." not in path.parts and "\\" not in name and ":" not in name
            and (name in {"pyproject.toml", "requirements.txt", "README.md", "LICENSE"}
                 or (name.startswith("src/") and name.endswith(".py"))
                 or (name.startswith("configs/") and path.suffix in {".json", ".md"})))

def install_bundle(archive_path: str | Path, target_root: str | Path, expected_sha256: str) -> tuple[Path, dict]:
    archive_path, target_root = Path(archive_path), Path(target_root).resolve()
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    if not expected_sha256 or digest != expected_sha256:
        raise ValueError("Gói source không đúng SHA-256 của notebook")
    with zipfile.ZipFile(archive_path) as archive:
        entries = archive.infolist()
        names = [row.filename for row in entries]
        if len(names) != len(set(names)) or len(names) > 150 or sum(row.file_size for row in entries) > 8 * 1024**2:
            raise ValueError("Gói source có cấu trúc hoặc dung lượng không hợp lệ")
        manifest = json.loads(archive.read("bundle_manifest.json"))
        if manifest.get("format_version") != 1 or manifest.get("runtime_protocol") != 1:
            raise ValueError("Gói source khác phiên bản runtime")
        if set(names) != set(manifest["files"]) | {"bundle_manifest.json"}:
            raise ValueError("Danh sách file không khớp manifest")
        for row in entries:
            if row.filename != "bundle_manifest.json" and not allowed_path(row.filename):
                raise ValueError("File không thuộc source allowlist")
            if stat.S_ISLNK(row.external_attr >> 16) or row.flag_bits & 1:
                raise ValueError("Gói source chứa symlink hoặc file mã hóa")
        contents = {name: archive.read(name) for name in names}
        for name, expected in manifest["files"].items():
            if hashlib.sha256(contents[name]).hexdigest() != expected:
                raise ValueError("Hash file không khớp: " + name)
    target_root.mkdir(parents=True, exist_ok=True)
    destination = target_root / digest[:16]
    if destination.exists():
        if destination.is_symlink() or not destination.resolve().is_relative_to(target_root):
            raise ValueError("Runtime source path is a symlink or outside the target root")
        expected_python = {name for name in manifest["files"] if name.startswith("src/") and name.endswith(".py")}
        actual_python = {p.relative_to(destination).as_posix() for p in (destination / "src").rglob("*.py")}
        if actual_python != expected_python:
            raise ValueError("Runtime source contains unexpected Python files")
        if any(not (destination / name).is_file() or (destination / name).is_symlink()
               or hashlib.sha256((destination / name).read_bytes()).hexdigest() != expected
               for name, expected in manifest["files"].items()):
            raise ValueError("Source đã cài có thay đổi; chọn thư mục source mới")
    else:
        with tempfile.TemporaryDirectory(prefix=".source-", dir=target_root) as temporary:
            staging = Path(temporary).resolve()
            if not staging.is_relative_to(target_root) or not destination.resolve().is_relative_to(target_root):
                raise ValueError("Source destination escapes runtime root")
            unpacked = staging / "runtime"
            for name, data in contents.items():
                target = unpacked / name
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(data)
            unpacked.rename(destination)
    return destination, manifest

source_root, source_manifest = install_bundle('/content/arxiv_parser_runtime.zip', '/content/parser-source', SOURCE_SHA256)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(source_root)], check=True)
os.chdir(source_root)
root = Path('/content/parser-submission')
if root.exists():
    raise RuntimeError('Output đã có; dùng runtime mới để có lượt benchmark không dùng cache.')
root.mkdir()
with zipfile.ZipFile('/content/arxiv_parser_input.zip') as archive:
    members = archive.infolist()
    if len(members)>5000 or sum(m.file_size for m in members)>256*1024**2:
        raise RuntimeError('Gói input vượt kích thước dự kiến.')
    for member in members:
        target = root/member.filename
        if not target.resolve().is_relative_to(root.resolve()) or stat.S_ISLNK(member.external_attr>>16):
            raise RuntimeError('Đường dẫn input không hợp lệ.')
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))
print('Input:', root)


In [ ]:
out = root/'parsed'
out.mkdir()
command = ['/usr/bin/time', '-v', '-o', str(out/'time_v.txt'), sys.executable, '-u', '-m', 'src.parser.main',
           '--publications', str(root/'m1/publications.parquet'), '--raw-dir', str(root/'raw'),
           '--output-dir', str(out), '--root-overrides', str(source_root/'configs/submission_parser_roots_20261106.json'),
           '--sample-interval-sec', '0.5', '--min-free-gb', '1']
(out/'run_configuration.json').write_text(json.dumps({'command':command,'source_sha256':SOURCE_SHA256,
    'input_sha256':INPUT_SHA256,'source_manifest':source_manifest,'scope':'73 papers / 119 versions'},indent=2))
with (out/'parser.log').open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end=''); log.write(line); log.flush()
    code = process.wait()
if code:
    raise RuntimeError('Parser exit code: '+str(code))


In [ ]:
subprocess.run([sys.executable,'-m','src.parser.validation','--course-dir',str(root/'course'),
                '--parsed-dir',str(out),'--publications',str(root/'m1/publications.parquet'),
                '--output',str(out/'project_validation.json')],check=True)
stats=json.loads((out/'stats.json').read_text())
print('Môi trường:',stats['environment'])
print('Trạng thái:',stats['status_counts'])
print('Wall time:',stats['runtime_sec'],'giây')
import shutil
result=shutil.make_archive('/content/parser_colab_results','zip',root_dir=out)
files.download(result)
